In [132]:
import torch
from transformers import  CamembertForMaskedLM, CamembertTokenizer, CamembertModel,RobertaTokenizer,RobertaModel,RobertaForMaskedLM, pipeline
from transformers import BertForMaskedLM, BertTokenizer, BertModel, AlbertConfig, AlbertModel, AlbertTokenizer, AlbertForMaskedLM 
from transformers import TFAutoModel, AutoModelForMaskedLM, AutoTokenizer, AutoModelForCausalLM
import math
import numpy as np
import json
import random
import pandas as pd
import tensorflow as tf
from torch.nn import CrossEntropyLoss
from tqdm.notebook import tqdm
import re
from scipy.optimize import linear_sum_assignment

In [133]:
import logging
import os
import sys
logging.basicConfig(level=logging.INFO)
import warnings
warnings.filterwarnings('ignore')

In [134]:
#Data
source = "test/Polish.txt"
file = open(source, "r", encoding = 'utf-8')
lines = file.readlines()

data = []

for l in range(12):
    line = lines[l]
    
    data.append(line)

In [135]:
def bert_predict(text, model, tokenizer):
    # Tokenized input
    # text = "[CLS] I got restricted because Tom reported my reply [SEP]"
    text = "[CLS] " + text + " [SEP]" #special token for BERT, RoBERTa
    tokenized_text = tokenizer.tokenize(text)
    sentence_score = 0
    length = len(tokenized_text)-2
    for masked_index in range(1,len(tokenized_text)-1):
        # Mask a token that we will try to predict back with `BertForMaskedLM`
        masked_word = tokenized_text[masked_index]
        #tokenized_text[masked_index] = '<mask>' #special token for XLNet
        tokenized_text[masked_index] = '[MASK]' #special token for BERT, RoBerta
        # Convert token to vocabulary indices
        indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
        index = torch.tensor(tokenizer.convert_tokens_to_ids(masked_word))
        tokens_tensor = torch.tensor([indexed_tokens])
        tokens_tensor = tokens_tensor.to('cuda')
        index = index.to('cuda')
        #masked_tensor = torch.tensor([masked_index])
        with torch.no_grad():
            outputs = model(tokens_tensor.to('cuda'))
        prediction_scores = outputs[0]
        prediction_scores = prediction_scores.view(-1, model.config.vocab_size)
        prediction_scores = prediction_scores[masked_index].unsqueeze(0)
        loss_fct = CrossEntropyLoss(ignore_index=-1)  # -1 index = padding token
        masked_lm_loss = loss_fct(prediction_scores, index.view(-1))
        tokenized_text[masked_index] = masked_word
        sentence_score -= masked_lm_loss.item()
        tokenized_text[masked_index] = masked_word
    sentence_score = sentence_score/length
    return sentence_score

In [136]:
def uni_predict(text, model, tokenizer):
    # Tokenized input
    # text = "[CLS] I got restricted because Tom reported my reply [SEP]"
    text = text
    tokenized_text = tokenizer.tokenize(text)
    sentence_score = 0
    indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
    length = len(tokenized_text)
    tokens_tensor = torch.tensor([indexed_tokens])
    tokens_tensor = tokens_tensor.to('cuda')
    #masked_tensor = torch.tensor([masked_index])
    with torch.no_grad():
        outputs = model(tokens_tensor, labels= tokens_tensor)
    loss = outputs[0]
    sentence_score = -loss
    return sentence_score

In [155]:
def greedy_select(df):
    selected_positions = []
    remaining_rows = set(df.index)
    remaining_columns = set(df.columns)

    while len(remaining_rows) > 0 and len(remaining_columns) > 0:
        min_value = float('inf')
        min_position = None

        # Find the smallest value and its position
        for row in remaining_rows:
            for column in remaining_columns:
                value = df.at[row, column]
                if value < min_value:
                    min_value = value
                    min_position = (row, column)

        # Remove the row and column
        remaining_rows.remove(min_position[0])
        remaining_columns.remove(min_position[1])

        # Add the position to the selected list
        selected_positions.append(min_position)

    return sorted(selected_positions, key=lambda x: x[0])

In [156]:
def score_model(model, tokenizer, data):
    opts = ["stały","dzieci","dalszy","trudno","bawi","krajach","pieniądze","kosztowne","młode","wierzą","osiąga","trenowania"]#, "stały","dzieci"]
    #opts = ["vyrástli","deti","druhoradé","ťažké","hrá","krajinách","peniaze","drahé","nevyvinuté","veria","dostane","trénovania"]
    df = pd.DataFrame()
    t1_score = 0
    t3_score = 0
    for d in tqdm(data):
        correct = opts[data.index(d)]
        print(d)
        print("Correct option is: ", correct)
        scores = {}
        for o in opts:
            sentence = d.replace("{}", o)
            scores.update({o : float(uni_predict(sentence, model, tokenizer).item())})
        df = df.append(scores, ignore_index=True)
        scores = sorted(scores.items(), key=lambda x: x[1], reverse = True)
        i = 0
        for key, value in scores:
            print(key, ':', value)
            t1_score += ((i == 0) and (key == correct))
            t3_score += ((i < 3) and (key == correct))
            i += 1
        print()
    
    print("Top 1 Score:", t1_score/12)
    print("Top 3 Score:", t3_score/12)
    
    #print(df)
    df = df.apply(lambda row: row / row.mean(), axis=1)
    x,y = linear_sum_assignment(df)
    out = pd.DataFrame({'Word': df.columns[y], 'Sentence': df.index[x]})
    #print(out)
    final_score = 0
    for n in range(12):
        final_score += (opts[n] == out.iloc[n]['Word'])
    final_score = final_score/12
    print("Forced Choiced Combinatorially Optimized Score:",final_score)
    greedy_values = greedy_select(df) 
    #print("Greedy Min Values:",greedy_values)
    final_score = 0
    for n in range(12):
        final_score += (opts[n] == greedy_values[n][1])
    final_score = final_score/12
    print("Forced Choiced Greedy Algorithm Score:",final_score)

In [157]:
#Polish
#print(torch.cuda.is_available())
model = BertForMaskedLM.from_pretrained("dkleczek/bert-base-polish-uncased-v1",ignore_mismatched_sizes=True).cuda()
tokenizer = BertTokenizer.from_pretrained("dkleczek/bert-base-polish-uncased-v1")
#print(bert_predict("text text text", model, tokenizer))
scoredModel = score_model(model, tokenizer, data)

Some weights of the model checkpoint at dkleczek/bert-base-polish-uncased-v1 were not used when initializing BertForMaskedLM: ['cls.seq_relationship.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


  0%|          | 0/12 [00:00<?, ?it/s]

Rodzice, których dzieci wykazują szczególne zainteresowanie jakimś rodzajem sportu, mają do podjęcia trudną decyzję. Czy powinni oni pozwolić swoim dzieciom na trenowanie, by {} się one najlepszymi sportowcami? 

Correct option is:  stały
stały : -0.1063566654920578
osiąga : -0.11348798871040344
trenowania : -0.27235785126686096
bawi : -0.28937220573425293
dzieci : -0.39617660641670227
wierzą : -0.48057010769844055
pieniądze : -0.5180937051773071
młode : -0.5759477615356445
krajach : -0.6266823410987854
kosztowne : -0.6311476230621338
trudno : -0.646591305732727
dalszy : -0.8302435874938965

Dla wielu {} to oznacza rozpoczęcie w bardzo młodym wieku.

Correct option is:  dzieci
młode : -0.22091928124427795
dzieci : -0.23963668942451477
trenowania : -0.2647949457168579
pieniądze : -0.2696557641029358
krajach : -0.2925722599029541
osiąga : -0.3072097599506378
wierzą : -0.34755390882492065
trudno : -0.6901698708534241
dalszy : -0.911700963973999
kosztowne : -1.0733739137649536
bawi : -1.18

In [158]:
#Czech 2
#Uses BERT tokenizer to avoid sentencepiece "not a string" error
tokenizer = BertTokenizer.from_pretrained("UWB-AIR/Czert-A-base-uncased", from_tf=True)
model = AlbertForMaskedLM.from_pretrained("UWB-AIR/Czert-A-base-uncased", from_tf=True).cuda()
scoredModel = score_model(model, tokenizer, data)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'AlbertTokenizer'. 
The class this function is called from is 'BertTokenizer'.
All TF 2.0 model weights were used when initializing AlbertForMaskedLM.

Some weights of AlbertForMaskedLM were not initialized from the TF 2.0 model and are newly initialized: ['predictions.decoder.weight', 'predictions.decoder.bias', 'predictions.decoder.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/12 [00:00<?, ?it/s]

Rodzice, których dzieci wykazują szczególne zainteresowanie jakimś rodzajem sportu, mają do podjęcia trudną decyzję. Czy powinni oni pozwolić swoim dzieciom na trenowanie, by {} się one najlepszymi sportowcami? 

Correct option is:  stały
trudno : -0.6725392937660217
bawi : -0.6728445291519165
kosztowne : -0.6744073629379272
krajach : -0.6801846027374268
trenowania : -0.6827566027641296
pieniądze : -0.6921865344047546
dalszy : -0.6976032257080078
młode : -0.6997364163398743
osiąga : -0.7263249754905701
stały : -0.7269824743270874
dzieci : -0.7314903140068054
wierzą : -0.7504576444625854

Dla wielu {} to oznacza rozpoczęcie w bardzo młodym wieku.

Correct option is:  dzieci
pieniądze : -1.0340195894241333
stały : -1.0980368852615356
wierzą : -1.0980420112609863
dalszy : -1.1636160612106323
dzieci : -1.1662133932113647
kosztowne : -1.1724653244018555
trudno : -1.2041248083114624
osiąga : -1.2123011350631714
krajach : -1.2469156980514526
bawi : -1.2576687335968018
trenowania : -1.27082991

In [159]:
tokenizer = RobertaTokenizer.from_pretrained('gerulata/slovakbert')
model = RobertaForMaskedLM.from_pretrained('gerulata/slovakbert').cuda()
scoredModel = score_model(model, tokenizer, data)

  0%|          | 0/12 [00:00<?, ?it/s]

Rodzice, których dzieci wykazują szczególne zainteresowanie jakimś rodzajem sportu, mają do podjęcia trudną decyzję. Czy powinni oni pozwolić swoim dzieciom na trenowanie, by {} się one najlepszymi sportowcami? 

Correct option is:  stały
bawi : -0.5924400091171265
stały : -0.6041882634162903
kosztowne : -0.6041926741600037
pieniądze : -0.6158885955810547
dalszy : -0.6219311952590942
trudno : -0.6244688034057617
dzieci : -0.6308415532112122
trenowania : -0.6529825925827026
wierzą : -0.6578115820884705
młode : -0.677984893321991
osiąga : -0.7316378355026245
krajach : -0.7799386978149414

Dla wielu {} to oznacza rozpoczęcie w bardzo młodym wieku.

Correct option is:  dzieci
pieniądze : -1.0840263366699219
dalszy : -1.1504778861999512
osiąga : -1.1515871286392212
wierzą : -1.1554813385009766
dzieci : -1.1919898986816406
trudno : -1.2038441896438599
młode : -1.2200614213943481
krajach : -1.2207918167114258
kosztowne : -1.2214614152908325
bawi : -1.2398371696472168
trenowania : -1.277229666

In [160]:
tokenizer = AutoTokenizer.from_pretrained("Milos/slovak-gpt-j-1.4B")
model = AutoModelForCausalLM.from_pretrained("Milos/slovak-gpt-j-1.4B").cuda()
scoredModel = score_model(model, tokenizer, data)

  0%|          | 0/12 [00:00<?, ?it/s]

Rodzice, których dzieci wykazują szczególne zainteresowanie jakimś rodzajem sportu, mają do podjęcia trudną decyzję. Czy powinni oni pozwolić swoim dzieciom na trenowanie, by {} się one najlepszymi sportowcami? 

Correct option is:  stały
stały : -2.7270100116729736
wierzą : -2.81195068359375
pieniądze : -2.8188555240631104
dzieci : -2.83113694190979
trudno : -2.8629279136657715
kosztowne : -2.8890092372894287
młode : -2.88948917388916
trenowania : -2.8930814266204834
dalszy : -2.927680015563965
osiąga : -2.9296674728393555
bawi : -2.931673526763916
krajach : -3.0209734439849854

Dla wielu {} to oznacza rozpoczęcie w bardzo młodym wieku.

Correct option is:  dzieci
młode : -3.271533250808716
dzieci : -3.332037925720215
pieniądze : -3.395015239715576
osiąga : -3.411688804626465
kosztowne : -3.439859628677368
trenowania : -3.46828293800354
stały : -3.4966259002685547
dalszy : -3.5042569637298584
wierzą : -3.5457377433776855
trudno : -3.628953695297241
bawi : -3.7811663150787354
krajach :